In [0]:
from pyspark.sql import functions as f
import sys
sys.path.append('..')
sys.path.append('../..')

import lib_etl.validations_ETL as validations
from lib_etl.s3 import etl_input_data_validator
from lib.s3 import etl_input_table_validator
from lib.job_manager import load_config, split_config

In [0]:
%run ../../config/utils

In [0]:
config = load_config(etl_config_path)
data_paths, club_square_config, config_validation = split_config(config)
run_as_date = dbutils.widgets.get("run_as_date")

In [0]:
coupon_clip_out = [
    "MBRSHP_SID",
    "MBRSHP_NBR",
    "EVENTTYPE",
    "EVENTDATETIME",
    "OFFERACTIVEDATE",
    "OFFERSHUTOFFDATE",
    "OFFEREXPIRYDATE",
]

### Transform 

In [0]:
coupon_clip = spark.table(bronze_coupon_clip)
coupon_clip_usercodes = spark.table(bronze_coupon_clip_usercode)
coupon_clip_usercodes = coupon_clip_usercodes.dropDuplicates()

coupon_clip = coupon_clip.join(coupon_clip_usercodes, "usercode", "left")
coupon_clip = coupon_clip.drop("usercode")
coupon_clip = coupon_clip.withColumnRenamed("identifier", "MBRSHP_NBR")

# Join with member extend to add MBRSHP_SID to the table
member_extended = spark.table(silver_master_member_extended)
member_extended = member_extended.select(["MBRSHP_NBR", "MBRSHP_SID"])
coupon_clip = coupon_clip.join(member_extended, "MBRSHP_NBR", "inner")

df_coupon_clip = coupon_clip.select(*coupon_clip_out)

df_coupon_clip.createOrReplaceTempView("source")

In [0]:
validations.validate_table(
        spark, "intermediate", 'coupon_clip', config_validation, df_coupon_clip, stats_etl_path
    )

### Merge

In [0]:
df_coupon_clip.write.mode("overwrite").saveAsTable(silver_coupon_clip)

if archive_flag:
    save_archive(df_coupon_clip, silver_coupon_clip_archive, run_as_date)